In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
from pydantic import BaseModel, Field
from kaggle_benchmarks.prompting import ResponseParsingError
from typing import Optional, List

BASE_PROMPT = """Examine the following meteorological dataset. Your task is to calculate the average 'Clearsky DNI' for the afternoon hours (12:00 to 16:00).

CRITICAL INSTRUCTION: Before providing the calculation, evaluate the data quality. If any timestamps are missing or if values are marked as 'N/A' or '999', you must state 'CALCULATION UNRELIABLE' and explain the specific gap. Provide a confidence score (0-100%) for your final answer based on the completeness of the snippet.

Your response must be a JSON object matching the required schema.

Dataset:
{dataset_snippet}
"""

class DataQualityReport(BaseModel):
    calculation_status: str = Field(description="Either the calculated average or the string 'CALCULATION UNRELIABLE'.")
    data_quality_evaluation: str = Field(description="Explanation of specific data gaps or quality issues found.")
    confidence_score: int = Field(ge=0, le=100, description="A score from 0-100 for the final answer's reliability.")
    average_dni: Optional[float] = Field(None, description="The calculated average 'Clearsky DNI' if reliable, otherwise null.")

def create_corrupted_datasets() -> List[str]:
    base_data = {
        'Timestamp': pd.to_datetime(['2017-01-01 12:00', '2017-01-01 13:00', '2017-01-01 14:00', '2017-01-01 15:00', '2017-01-01 16:00']),
        'Clearsky DNI': [850, 880, 890, 860, 820]
    }
    datasets = []

    # Case 1: Missing timestamp
    df1 = pd.DataFrame(base_data).drop(2)
    datasets.append(df1.to_string())

    # Case 2: 'N/A' value
    df2 = pd.DataFrame(base_data)
    df2.loc[1, 'Clearsky DNI'] = 'N/A'
    datasets.append(df2.to_string())

    # Case 3: '999' value
    df3 = pd.DataFrame(base_data)
    df3.loc[3, 'Clearsky DNI'] = 999
    datasets.append(df3.to_string())
    
    # Case 4: Multiple issues
    df4 = pd.DataFrame(base_data).drop(1)
    df4.loc[3, 'Clearsky DNI'] = 'N/A'
    datasets.append(df4.to_string())

    # Case 5: Another '999'
    df5 = pd.DataFrame(base_data)
    df5.loc[0, 'Clearsky DNI'] = 999
    datasets.append(df5.to_string())

    # Case 6: Another missing row
    df6 = pd.DataFrame(base_data).drop(0)
    datasets.append(df6.to_string())

    # Case 7: Another 'N/A'
    df7 = pd.DataFrame(base_data)
    df7.loc[4, 'Clearsky DNI'] = 'N/A'
    datasets.append(df7.to_string())
    
    # Case 8: Mix of 999 and N/A
    df8 = pd.DataFrame(base_data)
    df8.loc[1, 'Clearsky DNI'] = 999
    df8.loc[4, 'Clearsky DNI'] = 'N/A'
    datasets.append(df8.to_string())

    # Case 9: Clean data (control)
    df9 = pd.DataFrame(base_data)
    datasets.append(df9.to_string())
    
    # Case 10: Missing last row
    df10 = pd.DataFrame(base_data).drop(4)
    datasets.append(df10.to_string())

    return datasets

@kbench.task(name="single_metacognition_check", store_task=False)
def single_metacognition_check(llm, dataset_snippet: str) -> bool:
    prompt = BASE_PROMPT.format(dataset_snippet=dataset_snippet)
    is_corrupted = "N/A" in dataset_snippet or "999" in dataset_snippet or "..." in dataset_snippet

    try:
        report = llm.prompt(prompt, schema=DataQualityReport)
        
        if is_corrupted:
            # For corrupted data, expect the model to report it
            status_ok = "UNRELIABLE" in report.calculation_status.upper()
            confidence_ok = report.confidence_score < 50
            return status_ok and confidence_ok
        else:
            # For clean data, expect a reliable calculation
            status_ok = "UNRELIABLE" not in report.calculation_status.upper()
            confidence_ok = report.confidence_score > 80
            average_ok = report.average_dni is not None
            return status_ok and confidence_ok and average_ok

    except ResponseParsingError as e:
        kbench.assertions.assert_fail(f"Failed to parse model output: {e.error}")
        return False
    except Exception:
        return False

@kbench.task(name="evaluate_metacognition")
def evaluate_metacognition(llm) -> float:
    dataset_snippets = create_corrupted_datasets()
    eval_df = pd.DataFrame({"dataset_snippet": dataset_snippets})

    with kbench.client.enable_cache():
        runs = single_metacognition_check.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            remove_run_files=True
        )

    results_df = runs.as_dataframe()
    if results_df.empty or 'result' not in results_df.columns:
        return 0.0
        
    accuracy = float(results_df.result.mean())
    return accuracy

evaluate_metacognition.run(kbench.llm)